In [ ]:
from pop import Pilot, AI
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
import threading
import time

# pop 라이브러리의 Pilot 모듈에서 AutoCar 객체 생성
car = Pilot.AutoCar()

# 주행 상태를 관리하는 전역 변수
is_auto_driving = False

# 웹뷰 형식의 UI 위젯 생성
btn_auto = widgets.Button(description='자율주행 시작', button_style='success')
btn_manual = widgets.Button(description='직접주행 (조이스틱)', button_style='info')
btn_stop = widgets.Button(description='정지', button_style='danger')
camera_view = widgets.Image(format='jpeg', width=320, height=240)

# 버튼들을 가로로, 카메라 뷰를 아래로 배치
output_ui = widgets.VBox([
    widgets.HBox([btn_auto, btn_manual, btn_stop]),
    camera_view
])

# 카메라 객체 초기화 (기본 웹캠 0번 사용)
cap = cv2.VideoCapture(0)

def process_auto_drive():
    """왼편 하얀 선을 추종하여 자율주행을 수행하는 함수"""
    global is_auto_driving
    
    while is_auto_driving:
        ret, frame = cap.read()
        if not ret:
            continue
            
        height, width = frame.shape[:2]
        
        # 차체 왼편의 선을 추종하기 위해 화면의 '왼쪽 아래 사분면'을 ROI(관심 영역)로 설정
        roi = frame[height//2:height, 0:width//2]
        
        # 하얀 선 검출을 위한 그레이스케일 변환 및 이진화(Threshold) 적용
        gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        _, mask = cv2.threshold(gray, 200, 255, cv2.THRESH_BINARY)
        
        # 모멘트를 이용하여 하얀 선의 무게중심(Centroid) 계산
        M = cv2.moments(mask)
        if M['m00'] > 0:
            cx = int(M['m10'] / M['m00'])
            
            # 왼쪽 영역 내에서의 목표 지점 (왼쪽 화면의 정중앙)
            target_x = (width // 2) // 2
            
            # 목표 지점과 현재 차선의 중심 간의 오차 계산
            error = cx - target_x
            
            # 오차를 기반으로 조향값 계산 (-1.0 ~ 1.0 범위로 정규화)
            steering_val = error / target_x
            steering_val = max(min(steering_val, 1.0), -1.0) 
            
            # AutoCar 제어
            car.steering = steering_val
            car.forward(40) # 적절한 속도로 전진
        else:
            # 선이 보이지 않으면 차량 정지
            car.stop()
            
        # UI 카메라 위젯에 결과 프레임 업데이트
        cv2.rectangle(frame, (0, height//2), (width//2, height), (0, 255, 0), 2)
        _, jpeg = cv2.imencode('.jpg', frame)
        camera_view.value = jpeg.tobytes()
        
        time.sleep(0.05)

def on_auto_clicked(b):
    """자율주행 버튼 클릭 이벤트"""
    global is_auto_driving
    if not is_auto_driving:
        is_auto_driving = True
        clear_output(wait=True)
        display(output_ui)
        # 메인 UI 블로킹을 막기 위해 데몬 스레드에서 주행 루프 실행
        threading.Thread(target=process_auto_drive, daemon=True).start()

def on_manual_clicked(b):
    """직접주행 버튼 클릭 이벤트"""
    global is_auto_driving
    is_auto_driving = False
    car.stop()
    
    clear_output(wait=True)
    display(output_ui)
    
    # 조이스틱 UI 렌더링
    car.joystick()

def on_stop_clicked(b):
    """정지 버튼 클릭 이벤트"""
    global is_auto_driving
    is_auto_driving = False
    car.stop()

# 각 버튼에 클릭 콜백 함수 연결
btn_auto.on_click(on_auto_clicked)
btn_manual.on_click(on_manual_clicked)
btn_stop.on_click(on_stop_clicked)

# 초기 UI 출력
display(output_ui)

In [ ]:
from pop import Pilot
import cv2
import time
import threading

# 1. 차량 객체 생성
car = Pilot.AutoCar()

# 2. 제어 변수 설정 (이 값을 튜닝하여 부드러움을 조절하세요)
is_auto_driving = False
Kp = 0.5  # 조향 반응 계수 (값이 작을수록 부드럽게 반응함)
SPEED = 30 # 주행 속도

def process_auto_drive():
    """비례 제어를 적용한 부드러운 자율주행 루프"""
    global is_auto_driving
    cap = cv2.VideoCapture(0)
    
    while is_auto_driving:
        ret, frame = cap.read()
        if not ret: continue
            
        height, width = frame.shape[:2]
        roi = frame[height//2:height, 0:width//2] # 왼쪽 하단 ROI
        
        gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
        _, mask = cv2.threshold(gray, 180, 255, cv2.THRESH_BINARY)
        
        M = cv2.moments(mask)
        if M['m00'] > 0:
            cx = int(M['m10'] / M['m00'])
            target_x = (width // 2) // 2
            error = cx - target_x
            
            # 비례 제어 적용: error * Kp로 부드러운 조향값 계산
            steering_val = (error / target_x) * Kp
            car.steering = max(min(steering_val, 1.0), -1.0)
            car.forward(SPEED)
        else:
            car.stop()
        time.sleep(0.05)
    
    cap.release()
    car.stop()

# 3. 주행 제어 함수
def start_auto():
    global is_auto_driving
    is_auto_driving = True
    threading.Thread(target=process_auto_drive, daemon=True).start()
    print("자율주행 시작됨")

def stop_all():
    global is_auto_driving
    is_auto_driving = False
    car.stop()
    print("정지")

# 4. UI 실행 (pop 내장 조이스틱 활용)
# 수동 조종이 필요할 때 이 셀을 실행하세요
car.stop()
car.joystick()